# CEJA pipeline — Cd²⁺ / Pb²⁺ quantification with synthetic noise augmentation

**Single-notebook, Kaggle-ready reproduction** of *"Synthetic noise injection asymmetrically
affects machine-learning quantification of Cd²⁺ and Pb²⁺ in multi-scan cyclic voltammetry."*

This notebook consolidates the whole code package (data prep → noise injection → feature
extraction → Random Forest CV → statistics → feature importance → hyperparameter tuning →
leakage-safe check → simulation → figures) into one story that runs top-to-bottom.

**How to use on Kaggle**
1. Add your data as a Kaggle Dataset whose folder contains `CADMIUM/` and `TIMBAL/`.
2. The config cell auto-detects it under `/kaggle/input/...`. If not found, set `DATA_ROOT` by hand.
3. Run all. Heavy stages have toggles in the config cell — use `QUICK_TEST = True` for a fast dry run.

Outputs (CSVs + figures) are written to `/kaggle/working/` so plotting can be re-run without recomputing.

## 1. Config & reproducibility
All paths, seeds, model settings and the color palette live here. Toggle the run scope to trade
fidelity for runtime.

In [ ]:
import os, glob, sys, time, pickle, warnings
import numpy as np
warnings.filterwarnings("ignore")

# ---- master scope toggle -------------------------------------------------
QUICK_TEST = False        # True -> tiny/fast end-to-end dry run

# ---- per-stage scope -----------------------------------------------------
N_SEEDS_MAIN  = 20        # sweep / stats / feature importance / leakage
N_SEEDS_HP    = 5         # hyperparameter tuning (slow); runbook uses 5-10
HP_FAST       = False     # True -> tiny GridSearch grid (smoke check only)
RUN_FULL_SNR_SWEEP   = True   # needed for fig8/fig9 (6 SNR levels x 3 seeds)
RUN_MULTICLASSIFIER  = True   # needed for fig11 (RF/XGB/SVR/MLP)
RUN_SIM              = True   # synthetic-voltammogram mechanism study

ALL_SEEDS = [42, 7, 123, 2024, 88, 314, 1, 999, 55, 7777,
             13, 21, 34, 101, 202, 303, 404, 505, 606, 808]
if QUICK_TEST:
    N_SEEDS_MAIN, N_SEEDS_HP, HP_FAST = 2, 2, True
    RUN_FULL_SNR_SWEEP = False

# ---- data location (auto-detect Kaggle) ----------------------------------
def find_data_root():
    cands = []
    if os.path.isdir("/kaggle/input"):
        for d in glob.glob("/kaggle/input/**/CADMIUM", recursive=True):
            cands.append(os.path.dirname(d))
    # local fallback (edit if running off Kaggle):
    cands.append(r"C:\\Users\\alyak\\Documents\\S2\\S2 (SEMESTER 2)\\BISMILLAH THESIS\\DATA 2 KELAS")
    for c in cands:
        if os.path.isdir(os.path.join(c, "CADMIUM")) and os.path.isdir(os.path.join(c, "TIMBAL")):
            return c
    raise FileNotFoundError("Could not find a folder containing CADMIUM/ and TIMBAL/. "
                            "Set DATA_ROOT manually.")

DATA_ROOT = find_data_root()
print("DATA_ROOT =", DATA_ROOT)

# ---- output dirs ---------------------------------------------------------
OUT_BASE    = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
RESULTS_DIR = os.path.join(OUT_BASE, "results"); os.makedirs(RESULTS_DIR, exist_ok=True)
FIG_DIR     = os.path.join(OUT_BASE, "figs");    os.makedirs(FIG_DIR, exist_ok=True)
CACHE_PATH  = os.path.join(OUT_BASE, "data_cache.pkl")

# ---- acquisition / metal constants --------------------------------------
CD_CONCS = ['2 ppm','4 ppm','6 ppm','8 ppm','10 ppm','20 ppm','40 ppm','60 ppm','80 ppm',
            '100 ppm','200 ppm','400 ppm','600 ppm','800 ppm','1000 ppm']
PB_CONCS = ['2PPM','4PPM','6PPM','8PPM','10PPM','20PPM','40PPM','60PPM','80PPM',
            '100PPM','200PPM','400PPM','600PPM','800PPM','1000PPM']
CONC_VALUES = [2,4,6,8,10,20,40,60,80,100,200,400,600,800,1000]
PEAK_WINDOW = {'Cd': (-0.95, -0.75), 'Pb': (-0.50, -0.30)}
N_POINTS_PER_SCAN, N_USED_SCANS = 180, 9

# ---- model / CV settings (paper Sec 2.7-2.8) -----------------------------
BIN_EDGES = [0, 10, 30, 100, 500, 1100]            # strata + per-band MAPE
RF_PARAMS = dict(n_estimators=100, max_depth=10, min_samples_leaf=2, n_jobs=-1)
TEST_SNR  = 30

# ---- numpy trapezoid/trapz compatibility (numpy 1.x and 2.x) -------------
trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

# ---- single color palette (one source of truth) --------------------------
# Semantic noise palette (kept for noise-type plots) + general accents.
NOISE_PALETTE = {'gaussian':'#1a9850','student_t':'#fee08b','pink':'#fdae61','hetero':'#d73027'}
ACCENT = {'Cd':'#2E86AB','Pb':'#A23B72','baseline':'#555555','good':'#1a9850','bad':'#d73027'}

import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':110, 'savefig.bbox':'tight', 'axes.grid':True,
                     'grid.alpha':0.3, 'font.size':10})
print("Output ->", OUT_BASE)

## 2. Imports

In [ ]:
import pandas as pd
from scipy.signal import savgol_filter
from scipy.stats import skew as scipy_skew, t as student_t, ttest_rel, wilcoxon, t as t_dist
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split, cross_val_predict
from sklearn.inspection import permutation_importance
from sklearn.metrics import (r2_score, mean_absolute_error, mean_absolute_percentage_error,
                             make_scorer)
print("sklearn / scipy / pandas ready")

## 3. Data loading & sanity checks
Read the raw `.xlsx` files (10 scans each; scan-1 = conditioning, dropped → 9 scans of 180 points),
serialize to a cache, and verify counts / shapes / value ranges. **Real data only.**

In [ ]:
def load_one_file(filepath, n_used_scans=N_USED_SCANS, n_points=N_POINTS_PER_SCAN):
    # Columns alternate [E_s1,i_s1,E_s2,i_s2,...]; header on row index 1 (units row).
    try:
        df = pd.read_excel(filepath, header=1)
    except Exception:
        return []
    scans = []
    for s_idx in range(1, n_used_scans + 1):           # skip scan-1 (idx 0)
        col_E, col_i = 2 * s_idx, 2 * s_idx + 1
        if col_i >= df.shape[1]:
            break
        E = pd.to_numeric(df.iloc[:, col_E], errors='coerce').values
        y = pd.to_numeric(df.iloc[:, col_i], errors='coerce').values
        valid = ~(np.isnan(E) | np.isnan(y))
        if valid.sum() < n_points:
            continue
        E, y = E[valid][:n_points], y[valid][:n_points]
        if len(E) == n_points:
            scans.append((E.astype(np.float32), y.astype(np.float32)))
    return scans

def build_dataset_from_xlsx():
    rows, t0 = [], time.time()
    for metal, conc_dirs, subdir in [('Cd', CD_CONCS, 'CADMIUM'), ('Pb', PB_CONCS, 'TIMBAL')]:
        root, pw = os.path.join(DATA_ROOT, subdir), PEAK_WINDOW[metal]
        for conc_dir, conc_val in zip(conc_dirs, CONC_VALUES):
            conc_path = os.path.join(root, conc_dir)
            if not os.path.isdir(conc_path):
                print("  WARN missing", conc_path); continue
            for fp in sorted(glob.glob(os.path.join(conc_path, '*.xlsx'))):
                scans = load_one_file(fp)
                if len(scans) < 3:
                    continue
                rows.append(dict(metal=metal, conc=float(conc_val), pw=pw,
                                 file=os.path.basename(fp), scans=scans))
        print(f"  {metal}: {sum(1 for r in rows if r['metal']==metal)} files ({time.time()-t0:.0f}s)")
    return rows

if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, 'rb') as f: data = pickle.load(f)
    print("Loaded cache:", len(data), "files")
else:
    data = build_dataset_from_xlsx()
    with open(CACHE_PATH, 'wb') as f: pickle.dump(data, f)
    print("Built + cached:", len(data), "files ->", CACHE_PATH)

# ---- sanity checks -------------------------------------------------------
from collections import Counter
cnt = Counter((r['metal'], int(r['conc'])) for r in data)
assert len(data) > 0, "no data loaded"
assert all(len(r['scans']) >= 3 for r in data)
E0, y0 = data[0]['scans'][0]
assert E0.shape == (N_POINTS_PER_SCAN,) and y0.shape == (N_POINTS_PER_SCAN,)
print("metals:", sorted(set(r['metal'] for r in data)),
      "| concs:", sorted(set(int(r['conc']) for r in data)))
print("files per (metal,ppm) -> min/max:", min(cnt.values()), max(cnt.values()))
print("scan length:", len(E0), "| E range:", float(E0.min()), float(E0.max()))

In [ ]:
# A couple of representative voltammograms (real signals)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, metal in zip(axes, ['Cd', 'Pb']):
    samp = next(r for r in data if r['metal'] == metal and int(r['conc']) == 100)
    for E, y in samp['scans'][:3]:
        ax.plot(E, y, lw=0.8, color=ACCENT[metal], alpha=0.7)
    ax.axvspan(*samp['pw'], color=ACCENT[metal], alpha=0.10)
    ax.set_title(f"{metal} @ 100 ppm (3 scans)"); ax.set_xlabel("E (V vs Ag/AgCl)")
    ax.set_ylabel("current (uA)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "sample_signals.png"))
plt.savefig(os.path.join(FIG_DIR, "sample_signals.pdf")); plt.show()

## 4. Synthetic noise injection (paper Sec 2.6)
Peak-relative SNR (IUPAC). Four single distributions plus five combined ones; combined components
are each scaled by σ/√k so the combined signal keeps the target SNR. Applied **per scan, before
feature extraction**.

In [ ]:
def gaussian_sigma(y, snr_db):
    n = len(y)
    coeffs = np.polyfit(np.arange(n), y, 2)
    detr = y - np.polyval(coeffs, np.arange(n))
    s, e = 70, min(110, n)
    ph = float(np.max(np.abs(detr[s:e]))) if e > s else 1.0
    if ph < 1e-3: ph = 1.0
    return ph / (10 ** (snr_db / 20.0))

def add_gaussian(y, snr_db, rng):
    return y + rng.normal(0, gaussian_sigma(y, snr_db), size=y.shape)

def add_student_t(y, snr_db, rng, df=1.5):
    sigma = gaussian_sigma(y, snr_db)
    raw = student_t.rvs(df, size=y.shape, random_state=rng)
    return y + (sigma / max(np.std(raw), 1e-3)) * raw

def add_pink(y, snr_db, rng, beta=3.0):
    n = len(y); sigma = gaussian_sigma(y, snr_db)
    freqs = np.fft.rfftfreq(n); freqs[0] = freqs[1]
    spectrum = 1.0 / (freqs ** (beta / 2.0))
    phases = np.exp(2j * np.pi * rng.uniform(size=len(freqs)))
    noise = np.fft.irfft(spectrum * phases, n=n)
    return y + noise / max(np.std(noise), 1e-3) * sigma

def add_heteroscedastic(y, snr_db, rng):
    sm = gaussian_sigma(y, snr_db)
    w = np.abs(y - np.median(y)); w = w / max(np.mean(w), 1e-3)
    return y + rng.normal(0, sm * (0.5 + 0.5 * w), size=y.shape)

NOISE_FNS = {'baseline': lambda y, snr, rng: y, 'gaussian': add_gaussian,
             'student_t': add_student_t, 'pink': add_pink, 'hetero': add_heteroscedastic}

def _gen_component(y, kind, sigma, rng):
    n = len(y)
    if kind == 'gaussian':  return rng.normal(0, sigma, size=n)
    if kind == 'student_t':
        raw = student_t.rvs(1.5, size=n, random_state=rng)
        return (sigma / max(np.std(raw), 1e-3)) * raw
    if kind == 'pink':
        freqs = np.fft.rfftfreq(n); freqs[0] = freqs[1]
        spectrum = 1.0 / (freqs ** 1.5)
        phases = np.exp(2j * np.pi * rng.uniform(size=len(freqs)))
        noise = np.fft.irfft(spectrum * phases, n=n)
        return noise / max(np.std(noise), 1e-3) * sigma
    if kind == 'hetero':
        w = np.abs(y - np.median(y)); w = w / max(np.mean(w), 1e-3)
        return rng.normal(0, sigma * (0.5 + 0.5 * w), size=n)
    raise ValueError(kind)

def make_combined(kinds):
    def _fn(y, snr_db, rng):
        sigma = gaussian_sigma(y, snr_db) / np.sqrt(len(kinds))
        noise = np.zeros_like(y, dtype=float)
        for k in kinds:
            noise = noise + _gen_component(y, k, sigma, rng)
        return y + noise
    return _fn

NOISE_FNS.update({'gaussian_pink': make_combined(['gaussian','pink']),
                  'gaussian_student_t': make_combined(['gaussian','student_t']),
                  'gaussian_hetero': make_combined(['gaussian','hetero']),
                  'pink_student_t': make_combined(['pink','student_t']),
                  'gaussian_pink_student_t': make_combined(['gaussian','pink','student_t'])})
print("noise functions:", list(NOISE_FNS))

## 5. Feature extraction & multi-scan aggregation (paper Sec 2.5)
7 per-scan peak features → aggregated to a 17-dim vector (mean + std of all 7, plus CV of
amp/pos/area). Savitzky-Golay smoothing (11,3); quadratic baseline outside the peak window.

In [ ]:
def split_forward_reverse(E, y):
    dE = np.diff(E)
    sc = np.where(np.diff(np.sign(dE)) != 0)[0]
    if len(sc) == 0:
        return (E, y), (np.array([]), np.array([]))
    t = sc[0] + 1
    return (E[:t+1], y[:t+1]), (E[t+1:], y[t+1:])

def extract_features_one_scan(E, y, peak_window):
    yf = savgol_filter(y, 11, 3)
    mo = (E < peak_window[0]) | (E > peak_window[1])
    baseline = (np.polyval(np.polyfit(E[mo], yf[mo], 2), E) if mo.sum() >= 4
                else np.full_like(yf, np.median(yf)))
    (Ef, yf_), (Er, yr) = split_forward_reverse(E, yf)
    (_, bf), (_, br) = split_forward_reverse(E, baseline)
    def amp_in_window(E_, y_):
        m = (E_ >= peak_window[0]) & (E_ <= peak_window[1])
        return np.ptp(y_[m]) if m.sum() > 0 else 0.0
    if amp_in_window(Er, yr) >= amp_in_window(Ef, yf_):
        Eu, yu, bu = Er, yr, br
    else:
        Eu, yu, bu = Ef, yf_, bf
    mask = (Eu >= peak_window[0]) & (Eu <= peak_window[1])
    if mask.sum() < 3:
        return None
    Ew, yw, bw = Eu[mask], yu[mask], bu[mask]
    yc = yw - bw
    ip = np.argmax(np.abs(yc)); A = float(yc[ip])
    half = abs(A) / 2.0
    above = (yc >= half) if A > 0 else (yc <= -half)
    if above.sum() >= 2:
        idxs = np.where(above)[0]
        sigma = float(abs(Ew[idxs[-1]] - Ew[idxs[0]])) / 2.355
    else:
        sigma = 0.0
    area = float(trapz(np.abs(yc), Ew))
    sk = float(scipy_skew(yc)) if len(yc) > 3 and np.std(yc) > 0 else 0.0
    noise_proxy = np.std(yu) if len(yu) > 5 else 1.0
    return {'amp': A, 'pos': float(Ew[ip]), 'wid': sigma, 'area': area, 'skew': sk,
            'snr': abs(A) / max(float(noise_proxy), 1e-3), 'bl': float(bw[ip])}

def aggregate_multiscan(scan_features):
    df = pd.DataFrame(scan_features); agg = {}
    for col in df.columns:
        agg[f'{col}_mean'] = float(np.mean(df[col].values))
        agg[f'{col}_std']  = float(np.std(df[col].values))
    for col in ['amp', 'pos', 'area']:
        m = abs(agg[f'{col}_mean'])
        agg[f'{col}_cv'] = (agg[f'{col}_std'] / (m + 1e-6)) if m > 0 else 0.0
    return agg

def build_features(data, noise_fn, snr_db, seed):
    rng = np.random.default_rng(seed); rows = []
    for d in data:
        sf = []
        for E, y in d['scans']:
            f = extract_features_one_scan(E, noise_fn(y, snr_db, rng), d['pw'])
            if f is not None:
                sf.append(f)
        if len(sf) < 3:
            continue
        rows.append({'metal': d['metal'], 'conc': d['conc'], 'file': d['file'],
                     **aggregate_multiscan(sf)})
    out = pd.DataFrame(rows)
    return out

# quick check: clean baseline produces the expected 17 feature columns
_chk = build_features(data, NOISE_FNS['baseline'], 0, 42)
_feat = [c for c in _chk.columns if c not in ('metal','conc','file')]
assert len(_feat) == 17, f"expected 17 features, got {len(_feat)}"
print("feature columns (17):", _feat)

## 6. Random Forest evaluation with stratified 5-fold CV
`StandardScaler` is fit on the **training fold only** (no leakage), RF fit on train, predictions on
held-out test. Per-band MAPE; primary metric = MAPE_mid (20–100 ppm).

In [ ]:
def evaluate_rf(df_features, seed, rf_params=None):
    rf_params = rf_params or RF_PARAMS
    feat_cols = [c for c in df_features.columns if c not in ('metal','conc','file')]
    results = []
    for metal in ['Cd', 'Pb']:
        sub = df_features[df_features['metal'] == metal].copy()
        if len(sub) < 30:
            continue
        X, y = sub[feat_cols].values, sub['conc'].values
        bins = np.digitize(y, BIN_EDGES) - 1
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        y_pred = np.zeros_like(y, dtype=float)
        for tr, te in skf.split(X, bins):
            sc = StandardScaler().fit(X[tr])               # fit on train only
            rf = RandomForestRegressor(random_state=seed, **rf_params)
            rf.fit(sc.transform(X[tr]), y[tr])
            y_pred[te] = rf.predict(sc.transform(X[te]))
        y_pred = np.clip(y_pred, 0.1, None)
        rel = np.abs(y - y_pred) / y * 100
        low, mid, high = y < 20, (y >= 20) & (y < 100), y >= 100
        results.append(dict(metal=metal, R2=r2_score(y, y_pred),
                            MAPE=mean_absolute_percentage_error(y, y_pred) * 100,
                            MAPE_low=float(np.mean(rel[low])) if low.sum() else np.nan,
                            MAPE_mid=float(np.mean(rel[mid])) if mid.sum() else np.nan,
                            MAPE_high=float(np.mean(rel[high])) if high.sum() else np.nan))
    return results

for r in evaluate_rf(_chk, 42):
    print(f"  baseline {r['metal']}: R2={r['R2']:.3f}  MAPE_mid={r['MAPE_mid']:.1f}%  "
          f"MAPE_high={r['MAPE_high']:.1f}%")

## 7. Augmentation sweep (single + combined noise, multi-seed)
Baseline + 4 single + 5 combined distributions at SNR=30, across `N_SEEDS_MAIN` seeds →
`results.csv`. Optionally also runs the full 6-level SNR sweep (for fig8/fig9).

In [ ]:
SINGLE   = ['gaussian','student_t','pink','hetero']
COMBINED = ['gaussian_pink','gaussian_student_t','gaussian_hetero','pink_student_t','gaussian_pink_student_t']
seeds_main = ALL_SEEDS[:N_SEEDS_MAIN]

rows = []
def _run(noise, snr, seed):
    df = build_features(data, NOISE_FNS[noise], snr, seed)
    for m in evaluate_rf(df, seed):
        rows.append(dict(method='multiscan', noise_type=noise, snr_db=snr, seed=seed, **m))

t0 = time.time()
for s in seeds_main: _run('baseline', 0, s)
for noise in SINGLE + COMBINED:
    for s in seeds_main: _run(noise, TEST_SNR, s)
print(f"focal-SNR sweep done ({time.time()-t0:.0f}s)")

if RUN_FULL_SNR_SWEEP:
    snr_seeds = ALL_SEEDS[:3]
    for noise in SINGLE:
        for snr in [5, 10, 15, 20, 25]:          # 30 already covered above
            for s in snr_seeds: _run(noise, snr, s)
    print(f"full SNR sweep done ({time.time()-t0:.0f}s)")

results = pd.DataFrame(rows)
results.to_csv(os.path.join(RESULTS_DIR, "results.csv"), index=False)
print("results.csv rows:", len(results))
results.head()

## 8. Statistical analysis (paired, exact)
Seed-by-seed paired comparison of each noise condition vs the clean baseline at SNR=30:
paired t-test, **exact Wilcoxon**, 95% CIs (t + bootstrap), Cohen's dz, rank-biserial.
One-sided for Cd (expect improvement), two-sided for Pb. → `augmentation_stats.csv`.

In [ ]:
def _ci_t(x, a=0.05):
    x = np.asarray(x, float); n = len(x)
    if n < 2: return (np.nan, np.nan)
    se = x.std(ddof=1) / np.sqrt(n); h = t_dist.ppf(1 - a/2, n-1) * se
    return (x.mean() - h, x.mean() + h)
def _ci_boot(x, a=0.05, nb=10000, seed=20260602):
    x = np.asarray(x, float); n = len(x)
    if n < 2: return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    mu = x[rng.integers(0, n, size=(nb, n))].mean(axis=1)
    return (float(np.percentile(mu, 100*a/2)), float(np.percentile(mu, 100*(1-a/2))))
def _dz(d):
    d = np.asarray(d, float); sd = d.std(ddof=1)
    return float(d.mean()/sd) if sd > 0 else np.nan
def _rbc(d):
    d = np.asarray(d, float); nz = d[d != 0]
    if len(nz) == 0: return 0.0
    rk = pd.Series(np.abs(nz)).rank().values
    return float((rk[nz > 0].sum() - rk[nz < 0].sum()) / rk.sum())

df_res = pd.read_csv(os.path.join(RESULTS_DIR, "results.csv")).drop_duplicates(
    subset=['method','noise_type','snr_db','seed','metal'])
NOISE_TYPES = SINGLE + COMBINED
srows = []
for metal in ['Cd','Pb']:
    alt = 'less' if metal == 'Cd' else 'two-sided'
    base = df_res[(df_res.noise_type=='baseline') & (df_res.metal==metal)].set_index('seed')['MAPE_mid']
    for noise in NOISE_TYPES:
        cond = df_res[(df_res.noise_type==noise) & (df_res.snr_db==TEST_SNR) &
                      (df_res.metal==metal)].set_index('seed')['MAPE_mid']
        seeds = sorted(set(base.index) & set(cond.index))
        if len(seeds) < 2: continue
        b, c = base.loc[seeds].values, cond.loc[seeds].values
        diff = c - b; pct = diff / b * 100.0
        t_stat, p_t = ttest_rel(c, b, alternative=alt)
        try:
            w_stat, p_w = wilcoxon(c, b, alternative=alt, zero_method='wilcox', mode='exact')
        except (ValueError, TypeError):
            w_stat, p_w = wilcoxon(c, b, alternative=alt, zero_method='wilcox')
        srows.append(dict(metal=metal, noise_type=noise, snr_db=TEST_SNR, n_seeds=len(seeds),
                          baseline_MAPE_mid=float(b.mean()), noise_MAPE_mid=float(c.mean()),
                          delta_pct=float(pct.mean()),
                          delta_pct_CI_lo=_ci_t(pct)[0], delta_pct_CI_hi=_ci_t(pct)[1],
                          delta_abs=float(diff.mean()),
                          delta_abs_CI_lo_boot=_ci_boot(diff)[0], delta_abs_CI_hi_boot=_ci_boot(diff)[1],
                          p_value_ttest=float(p_t), p_value_wilcoxon_exact=float(p_w),
                          cohen_dz=_dz(diff), rank_biserial=_rbc(diff), alternative=alt))
stats = pd.DataFrame(srows)
stats.to_csv(os.path.join(RESULTS_DIR, "augmentation_stats.csv"), index=False)
show = ['metal','noise_type','n_seeds','baseline_MAPE_mid','noise_MAPE_mid','delta_pct',
        'delta_pct_CI_lo','delta_pct_CI_hi','p_value_wilcoxon_exact','cohen_dz']
print(stats[show].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

## 9. Feature importance — do dispersion (std/CV) features gain importance after augmentation?
Permutation importance (MAPE-scored, on a held-out split) + RF built-in (+ SHAP if available),
clean vs augmented, across seeds. Reports the importance **share** of mean/std/cv groups.

In [ ]:
try:
    import shap; HAVE_SHAP = True
except Exception:
    HAVE_SHAP = False
print("SHAP available:", HAVE_SHAP)

def group_of(col): return col.rsplit('_', 1)[1]

def importances_for(df_features, seed, metal):
    feat_cols = [c for c in df_features.columns if c not in ('metal','conc','file')]
    sub = df_features[df_features.metal == metal]
    X, y = sub[feat_cols].values, sub['conc'].values
    bins = np.digitize(y, BIN_EDGES) - 1
    Xtr, Xte, ytr, yte, _, _ = train_test_split(X, y, bins, test_size=0.30,
                                                random_state=seed, stratify=bins)
    sc = StandardScaler().fit(Xtr)
    rf = RandomForestRegressor(random_state=seed, **RF_PARAMS).fit(sc.transform(Xtr), ytr)
    perm = permutation_importance(rf, sc.transform(Xte), yte, n_repeats=10, random_state=seed,
                                  scoring='neg_mean_absolute_percentage_error', n_jobs=-1).importances_mean
    builtin = rf.feature_importances_
    if HAVE_SHAP:
        sv = shap.TreeExplainer(rf).shap_values(sc.transform(Xte), check_additivity=False)
        shap_imp = np.abs(sv).mean(axis=0)
    else:
        shap_imp = np.full(len(feat_cols), np.nan)
    return pd.DataFrame({'feature': feat_cols, 'group': [group_of(c) for c in feat_cols],
                         'perm': np.clip(perm, 0, None), 'builtin': builtin, 'shap': shap_imp})

fi_seeds = ALL_SEEDS[:N_SEEDS_MAIN]; long_rows = []
for seed in fi_seeds:
    dfc = build_features(data, NOISE_FNS['baseline'], 0, seed)
    dfa = build_features(data, NOISE_FNS['gaussian'], TEST_SNR, seed)
    for metal in ['Cd','Pb']:
        for cond, dff in [('clean', dfc), ('augmented', dfa)]:
            imp = importances_for(dff, seed, metal)
            imp.insert(0, 'seed', seed); imp.insert(0, 'condition', cond); imp.insert(0, 'metal', metal)
            long_rows.append(imp)
fi_long = pd.concat(long_rows, ignore_index=True)
fi_long.to_csv(os.path.join(RESULTS_DIR, "feature_importance.csv"), index=False)

grp = []
for (metal, cond, seed), g in fi_long.groupby(['metal','condition','seed']):
    tot = g['perm'].clip(lower=0).sum()
    if tot <= 0: continue
    for gp in ['mean','std','cv']:
        grp.append(dict(metal=metal, condition=cond, seed=seed, group=gp,
                        share=g.loc[g.group==gp,'perm'].clip(lower=0).sum()/tot))
gdf = pd.DataFrame(grp)
gdf.to_csv(os.path.join(RESULTS_DIR, "feature_importance_grouped.csv"), index=False)
print("Permutation-importance share (std+cv), clean -> augmented:")
for metal in ['Cd','Pb']:
    cl = gdf[(gdf.metal==metal)&(gdf.condition=='clean')&(gdf.group.isin(['std','cv']))]['share'].mean()
    au = gdf[(gdf.metal==metal)&(gdf.condition=='augmented')&(gdf.group.isin(['std','cv']))]['share'].mean()
    print(f"  {metal}: {cl:.3f} -> {au:.3f}")

## 10. Hyperparameter tuning (nested CV) — does the effect survive tuning?
Inner GridSearchCV selects params on train folds; outer folds measure the tuned score (unbiased).
Compares default (paper) vs tuned MAPE_mid; confirms the asymmetry persists. Slow — controlled by
`N_SEEDS_HP` / `HP_FAST`.

In [ ]:
if HP_FAST:
    PARAM_GRID = {'n_estimators':[100], 'max_depth':[10, None],
                  'min_samples_split':[2], 'min_samples_leaf':[1, 2]}
else:
    PARAM_GRID = {'n_estimators':[100,200,400], 'max_depth':[10,20,None],
                  'min_samples_split':[2,5,10], 'min_samples_leaf':[1,2]}
mape_scorer = make_scorer(mean_absolute_percentage_error, greater_is_better=False)

def nested_cv(df_features, seed, metal):
    feat_cols = [c for c in df_features.columns if c not in ('metal','conc','file')]
    sub = df_features[df_features.metal == metal]
    X, y = sub[feat_cols].values, sub['conc'].values
    bins = np.digitize(y, BIN_EDGES) - 1
    outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    yp_t, yp_d = np.zeros_like(y, float), np.zeros_like(y, float)
    for tr, te in outer.split(X, bins):
        sc = StandardScaler().fit(X[tr])
        Xtr, Xte = sc.transform(X[tr]), sc.transform(X[te])
        inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
        gs = GridSearchCV(RandomForestRegressor(random_state=seed, n_jobs=1), PARAM_GRID,
                          scoring=mape_scorer, cv=inner.split(Xtr, bins[tr]), n_jobs=-1)
        gs.fit(Xtr, y[tr]); yp_t[te] = gs.best_estimator_.predict(Xte)
        yp_d[te] = RandomForestRegressor(random_state=seed, **RF_PARAMS).fit(Xtr, y[tr]).predict(Xte)
    yp_t, yp_d = np.clip(yp_t, 0.1, None), np.clip(yp_d, 0.1, None)
    def mid(yp):
        m = (y >= 20) & (y < 100); return float(np.mean(np.abs(y[m]-yp[m])/y[m])*100) if m.sum() else np.nan
    return mid(yp_t), mid(yp_d)

hp_seeds = ALL_SEEDS[:N_SEEDS_HP]; hp_rows = []; t0 = time.time()
for seed in hp_seeds:
    dfc = build_features(data, NOISE_FNS['baseline'], 0, seed)
    dfa = build_features(data, NOISE_FNS['gaussian'], TEST_SNR, seed)
    for cond, dff in [('clean', dfc), ('augmented', dfa)]:
        for metal in ['Cd','Pb']:
            tuned, default = nested_cv(dff, seed, metal)
            hp_rows.append(dict(metal=metal, condition=cond, seed=seed,
                                default_MAPE_mid=default, tuned_MAPE_mid=tuned))
    print(f"  hp seed={seed} done ({time.time()-t0:.0f}s)")
hp = pd.DataFrame(hp_rows); hp.to_csv(os.path.join(RESULTS_DIR, "hyperparameter_tuning.csv"), index=False)
agg = hp.groupby(['metal','condition']).agg(default=('default_MAPE_mid','mean'),
                                            tuned=('tuned_MAPE_mid','mean')).reset_index()
print(agg.to_string(index=False, float_format=lambda v: f"{v:.2f}"))
for metal in ['Cd','Pb']:
    cl = agg[(agg.metal==metal)&(agg.condition=='clean')]['tuned'].values
    au = agg[(agg.metal==metal)&(agg.condition=='augmented')]['tuned'].values
    if len(cl) and len(au):
        print(f"  {metal}: tuned clean={cl[0]:.2f}%  tuned aug={au[0]:.2f}%  delta={(au[0]-cl[0])/cl[0]*100:+.1f}%")

## 11. Leakage-safe augmentation — three protocols under strict file-level CV
A: clean→clean, B: noisy→noisy (paper), C: noisy(train)→clean(test). One file = one feature vector
(no augment-before-split duplication); scaler/RF fit on train fold only. If B improves Cd but C does
not, the benefit is a property of noisy multi-scan acquisition, not a leakage artifact.

In [ ]:
KEY = ['metal','conc','file']
def aligned_features(noise, snr, seed):
    clean = build_features(data, NOISE_FNS['baseline'], 0, seed).set_index(KEY)
    noisy = build_features(data, NOISE_FNS[noise], snr, seed).set_index(KEY)
    common = clean.index.intersection(noisy.index)
    return clean.loc[common].reset_index(), noisy.loc[common].reset_index()

def run_protocols(clean_df, noisy_df, seed):
    feat = [c for c in clean_df.columns if c not in KEY]; out = []
    for metal in ['Cd','Pb']:
        cm = clean_df[clean_df.metal==metal].reset_index(drop=True)
        nm = noisy_df[noisy_df.metal==metal].reset_index(drop=True)
        assert (cm['file'].values == nm['file'].values).all()
        Xc, Xn, y = cm[feat].values, nm[feat].values, cm['conc'].values
        bins = np.digitize(y, BIN_EDGES) - 1
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        preds = {p: np.zeros_like(y, float) for p in ('A_clean','B_noisy','C_trainonly')}
        for tr, te in skf.split(Xc, bins):
            sA = StandardScaler().fit(Xc[tr])
            preds['A_clean'][te] = RandomForestRegressor(random_state=seed, **RF_PARAMS).fit(
                sA.transform(Xc[tr]), y[tr]).predict(sA.transform(Xc[te]))
            sB = StandardScaler().fit(Xn[tr])
            preds['B_noisy'][te] = RandomForestRegressor(random_state=seed, **RF_PARAMS).fit(
                sB.transform(Xn[tr]), y[tr]).predict(sB.transform(Xn[te]))
            sC = StandardScaler().fit(Xn[tr])
            preds['C_trainonly'][te] = RandomForestRegressor(random_state=seed, **RF_PARAMS).fit(
                sC.transform(Xn[tr]), y[tr]).predict(sC.transform(Xc[te]))
        for proto, yp in preds.items():
            yp = np.clip(yp, 0.1, None); m = (y >= 20) & (y < 100)
            out.append(dict(metal=metal, protocol=proto, seed=seed,
                            MAPE_mid=float(np.mean(np.abs(y[m]-yp[m])/y[m])*100)))
    return out

lk_rows = []
for seed in ALL_SEEDS[:N_SEEDS_MAIN]:
    cdf, ndf = aligned_features('gaussian', TEST_SNR, seed)
    lk_rows += run_protocols(cdf, ndf, seed)
lk = pd.DataFrame(lk_rows); lk.to_csv(os.path.join(RESULTS_DIR, "leakage_safe.csv"), index=False)
print(lk.groupby(['metal','protocol'])['MAPE_mid'].mean().round(2).to_string())

## 12. Multi-classifier confirmation (fig11)
Same 17-feature representation, four regressors (RF / XGBoost / SVR / MLP), clean vs Gaussian@30.
Tree models should keep the Cd benefit; SVR/MLP weaker. Skips XGBoost if not installed.

In [ ]:
mc_rows = []
if RUN_MULTICLASSIFIER:
    from sklearn.svm import SVR
    from sklearn.neural_network import MLPRegressor
    try:
        from xgboost import XGBRegressor; HAVE_XGB = True
    except Exception:
        HAVE_XGB = False
    def make_model(name, seed):
        if name == 'RF':  return RandomForestRegressor(random_state=seed, **RF_PARAMS)
        if name == 'XGBoost': return XGBRegressor(n_estimators=100, max_depth=10, learning_rate=0.1,
                                                  random_state=seed, n_jobs=-1, verbosity=0)
        if name == 'SVR': return SVR(kernel='rbf', C=1.0, gamma='scale', epsilon=0.1)
        if name == 'MLP': return MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500,
                                              early_stopping=True, random_state=seed)
        raise ValueError(name)
    def eval_model(df_features, name, metal, seed):
        feat = [c for c in df_features.columns if c not in ('metal','conc','file')]
        sub = df_features[df_features.metal == metal]
        if len(sub) < 30: return None
        X, y = sub[feat].values, sub['conc'].values; bins = np.digitize(y, BIN_EDGES) - 1
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        yp = np.zeros_like(y, float)
        for tr, te in skf.split(X, bins):
            sc = StandardScaler().fit(X[tr])
            yp[te] = make_model(name, seed).fit(sc.transform(X[tr]), y[tr]).predict(sc.transform(X[te]))
        yp = np.clip(yp, 0.1, None); m = (y >= 20) & (y < 100)
        return float(np.mean(np.abs(y[m]-yp[m])/y[m])*100)
    models = ['RF','SVR','MLP'] + (['XGBoost'] if HAVE_XGB else [])
    for seed in ALL_SEEDS[:3]:
        dfc = build_features(data, NOISE_FNS['baseline'], 0, seed)
        dfa = build_features(data, NOISE_FNS['gaussian'], TEST_SNR, seed)
        for name in models:
            for cond, dff in [('baseline', dfc), ('gaussian', dfa)]:
                for metal in ['Cd','Pb']:
                    v = eval_model(dff, name, metal, seed)
                    if v is not None:
                        mc_rows.append(dict(model=name, noise_type=cond, metal=metal, seed=seed, MAPE_mid=v))
    mc = pd.DataFrame(mc_rows); mc.to_csv(os.path.join(RESULTS_DIR, "multi_classifier_results.csv"), index=False)
    print(mc.groupby(['model','metal','noise_type'])['MAPE_mid'].mean().round(2).to_string())
else:
    print("RUN_MULTICLASSIFIER = False (skipped)")

## 13. Simulation study (mechanism) — does peak shape alone reproduce the asymmetry?
Synthetic voltammograms where only the peak shape varies (Cd-like = broad/bumpy, Pb-like =
sharp/clean). Same 17-feature pipeline + RF. Self-contained; controlled by `RUN_SIM`.

In [ ]:
if RUN_SIM:
    SN_N_POINTS, SN_SCANS = 180, 9
    SV_MIN, SV_MAX, SPEAK = -1.2, -0.2, -0.70
    SHALF, SBASE_SNR = 0.28, 40.0
    SCONC = np.array(CONC_VALUES, float); SBINS = [0,10,30,100,500,1100]
    SFEATS = ['A','Ep','sigma','area','skew','snr','baseline']; SCV = ['A','Ep','area']
    def _g(v, mu, s): return np.exp(-0.5*((v-mu)/s)**2)
    def _amask(V): return np.abs(V - SPEAK) <= SHALF
    def _bcorr(V, y):
        wg = ~_amask(V); c = np.polyfit(V[wg], y[wg], 2); bl = np.polyval(c, V); return y-bl, bl
    def _pheight(V, y):
        bc, _ = _bcorr(V, y); return max(bc[_amask(V)].max(), 1e-9)
    def lump_heights(shape, rng):
        b = shape['bumpiness']; base = np.array([1.0, b])
        return np.clip(base * (1.0 + 0.20*rng.standard_normal(2)), 0.0, None)
    def make_clean(V, amp, width, heights, skew, osep, oamp):
        mu = SPEAK; off = (1.15 + 0.5*skew)*width
        main = heights[0]*_g(V, mu, 0.50*width); decoy = heights[1]*_g(V, mu+off, 0.70*width)
        sig = amp*(main+decoy)
        if np.isfinite(osep): sig = sig + amp*oamp*_g(V, mu+osep, 0.60*width)
        return sig + 12.0*(V-SV_MIN) + 8.0*(V-SV_MIN)**2
    def gen_file(rng, conc, shape):
        V = np.linspace(SV_MIN, SV_MAX, SN_N_POINTS)
        amp = 3.0*conc**0.80*(1.0+0.04*rng.standard_normal())
        width = shape['width']*(1.0+0.03*rng.standard_normal()); heights = lump_heights(shape, rng)
        scans = np.empty((SN_SCANS, SN_N_POINTS))
        for s in range(SN_SCANS):
            cln = make_clean(V, amp, width, heights, shape['skew'],
                             shape.get('overlap_sep', np.nan), shape.get('overlap_amp', 0.0))
            scans[s] = cln + (_pheight(V, cln)/(10**(SBASE_SNR/20.0)))*rng.standard_normal(SN_N_POINTS)
        return V, scans
    def inject(V, scans, nt, snr, rng):
        out = np.empty_like(scans)
        for s in range(scans.shape[0]):
            cln = scans[s]; sig = _pheight(V, cln)/(10**(snr/20.0)); n = cln.shape[0]
            if nt == 'gaussian': z = rng.standard_normal(n)
            elif nt == 'student_t': z = rng.standard_t(1.5, size=n)
            elif nt == 'pink':
                w = rng.standard_normal(n); F = np.fft.rfft(w); f = np.fft.rfftfreq(n, 1.0)
                sc = np.ones_like(f); sc[1:] = f[1:]**(-1.5); sc[0] = 0.0; z = np.fft.irfft(F*sc, n=n)
            else:
                w = np.abs(cln-cln.mean())+1e-9; z = rng.standard_normal(n)*(w/np.sqrt(np.mean(w**2)))
            out[s] = cln + z/(np.std(z)+1e-12)*sig
        return out
    def feat_one(V, y):
        ys = savgol_filter(y, 11, 3); bc, bl = _bcorr(V, ys); m = _amask(V); Vm, bcm = V[m], bc[m]
        pk = int(np.argmax(bcm)); A = bcm[pk]; Ep = Vm[pk]; bap = bl[m][pk]
        half = A/2.0; ab = bcm >= half
        if ab.any():
            idx = np.where(ab)[0]; fw = abs(Vm[idx[-1]]-Vm[idx[0]]); sigma = fw/2.3548 if fw>0 else (SV_MAX-SV_MIN)/SN_N_POINTS
        else: sigma = (SV_MAX-SV_MIN)/SN_N_POINTS
        pos = np.clip(bcm, 0, None); area = trapz(pos, Vm); w = pos+1e-12
        m1 = np.sum(w*Vm)/np.sum(w); sd = np.sqrt(np.sum(w*(Vm-m1)**2)/np.sum(w))+1e-12
        sk = np.sum(w*(Vm-m1)**3)/np.sum(w)/sd**3; snr = A/(np.std(y-ys)+1e-12)
        return np.array([A, Ep, sigma, area, sk, snr, bap])
    def file_vec(V, scans):
        ps = np.vstack([feat_one(V, scans[s]) for s in range(scans.shape[0])])
        means, stds = ps.mean(0), ps.std(0); ci = [SFEATS.index(f) for f in SCV]
        return np.concatenate([means, stds, stds[ci]/(np.abs(means[ci])+1e-9)])
    def sim_dataset(shape, nfiles, seed, nt=None, snr=None):
        rng = np.random.default_rng(seed); X, y = [], []
        for conc in SCONC:
            for _ in range(nfiles):
                V, scans = gen_file(rng, conc, shape)
                if nt is not None: scans = inject(V, scans, nt, snr, rng)
                X.append(file_vec(V, scans)); y.append(conc)
        X = np.asarray(X); y = np.asarray(y); return X, y, np.digitize(y, SBINS[1:-1])
    def sim_eval(X, y, labels, seed):
        skf = StratifiedKFold(5, shuffle=True, random_state=seed)
        yp = np.clip(cross_val_predict(RandomForestRegressor(random_state=seed, **RF_PARAMS),
                                       X, y, cv=skf.split(X, labels), n_jobs=-1), 0.1, None)
        mid = (y >= 20) & (y <= 100)
        return float(np.mean(np.abs(y[mid]-yp[mid])/y[mid])*100)
    CD_LIKE = dict(width=0.15, bumpiness=0.9, skew=0.4, overlap_sep=0.22, overlap_amp=0.5)
    PB_LIKE = dict(width=0.045, bumpiness=0.0, skew=0.0, overlap_sep=np.nan, overlap_amp=0.0)
    nf = 12 if QUICK_TEST else 24; sseeds = ALL_SEEDS[:(2 if QUICK_TEST else 6)]
    sim_out = []
    for nm, shp in [('Cd_like', CD_LIKE), ('Pb_like', PB_LIKE)]:
        for sd in sseeds:
            Xc, yc, lc = sim_dataset(shp, nf, sd); cl = sim_eval(Xc, yc, lc, sd)
            Xn, yn, ln = sim_dataset(shp, nf, sd, 'gaussian', TEST_SNR); g = sim_eval(Xn, yn, ln, sd)
            sim_out.append(dict(shape=nm, seed=sd, clean=cl, gaussian=g, delta_pct=(g-cl)/cl*100))
    sim = pd.DataFrame(sim_out); sim.to_csv(os.path.join(RESULTS_DIR, "sim_anchors.csv"), index=False)
    print(sim.groupby('shape')[['clean','gaussian','delta_pct']].mean().round(2).to_string())
else:
    print("RUN_SIM = False (skipped)")

## 14. Figures (read from saved CSVs)
All figures load the persisted artifacts so they can be restyled without recomputation. Each is
exported as high-resolution PNG **and** vector PDF.

In [ ]:
def savefig(name):
    plt.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=200)
    plt.savefig(os.path.join(FIG_DIR, name + ".pdf")); plt.show()

df_res = pd.read_csv(os.path.join(RESULTS_DIR, "results.csv")).drop_duplicates(
    subset=['method','noise_type','snr_db','seed','metal'])
stats = pd.read_csv(os.path.join(RESULTS_DIR, "augmentation_stats.csv"))

# ---- fig10: augmentation bars (single + combined) at SNR=30 with significance ----
def stars(p): return '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
order = SINGLE + COMBINED
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, metal in zip(axes, ['Cd','Pb']):
    bl = df_res[(df_res.noise_type=='baseline')&(df_res.metal==metal)]['MAPE_mid'].mean()
    ax.axhline(bl, color='black', ls='--', alpha=0.7, label=f'clean baseline ({bl:.1f}%)')
    means, errs, cols = [], [], []
    for n in order:
        sub = df_res[(df_res.noise_type==n)&(df_res.snr_db==TEST_SNR)&(df_res.metal==metal)]['MAPE_mid']
        means.append(sub.mean()); errs.append(sub.std())
        cols.append(NOISE_PALETTE.get(n, '#888888'))
    ax.bar(range(len(order)), means, yerr=errs, color=cols, capsize=3, edgecolor='black', linewidth=0.5)
    for i, n in enumerate(order):
        d = (means[i]-bl)/bl*100
        row = stats[(stats.metal==metal)&(stats.noise_type==n)]
        s = stars(float(row['p_value_wilcoxon_exact'].iloc[0])) if len(row) else ''
        ax.text(i, means[i]+errs[i]+0.5, f"{d:+.0f}%\n{s}", ha='center', fontsize=8,
                fontweight='bold', color=ACCENT['good'] if d<-5 else ACCENT['bad'] if d>5 else 'black')
    ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('MAPE_mid (%)'); ax.set_title(f"{metal}: augmentation @ SNR=30 ({df_res.seed.nunique()} seeds)")
    ax.legend(fontsize=8)
plt.suptitle("Asymmetric noise augmentation: strong for Cd, absent for Pb")
plt.tight_layout(); savefig("fig10_augmentation")

In [ ]:
# ---- fig8/fig9: SNR robustness (only if the full SNR sweep was run) ----
SNR_LEVELS = [30,25,20,15,10,5]
have_full = df_res[(df_res.noise_type=='gaussian')]['snr_db'].nunique() > 1
if have_full:
    # fig9 heatmap (MAPE_mid) Cd & Pb
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    for ax, metal in zip(axes, ['Cd','Pb']):
        M = np.full((len(SINGLE), len(SNR_LEVELS)), np.nan)
        for i, n in enumerate(SINGLE):
            for j, s in enumerate(SNR_LEVELS):
                sub = df_res[(df_res.noise_type==n)&(df_res.snr_db==s)&(df_res.metal==metal)]['MAPE_mid']
                if len(sub): M[i, j] = sub.mean()
        im = ax.imshow(M, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=60 if metal=='Cd' else 35)
        ax.set_xticks(range(len(SNR_LEVELS))); ax.set_xticklabels(SNR_LEVELS)
        ax.set_yticks(range(len(SINGLE))); ax.set_yticklabels(SINGLE)
        ax.set_xlabel('SNR (dB)'); ax.set_title(f"{metal} MAPE_mid")
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                if not np.isnan(M[i, j]): ax.text(j, i, f"{M[i,j]:.0f}", ha='center', va='center', fontsize=8)
        plt.colorbar(im, ax=ax, fraction=0.045)
    plt.suptitle("MAPE heatmap: 4 noise x 6 SNR"); plt.tight_layout(); savefig("fig9_heatmap")
else:
    print("Full SNR sweep not present (RUN_FULL_SNR_SWEEP=False) -> skipping fig8/fig9.")

In [ ]:
# ---- fig11: multi-classifier (only if run) ----
mcp = os.path.join(RESULTS_DIR, "multi_classifier_results.csv")
if os.path.exists(mcp):
    mc = pd.read_csv(mcp); models = list(mc['model'].unique())
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    for ax, metal in zip(axes, ['Cd','Pb']):
        x = np.arange(len(models)); w = 0.35
        cm = [mc[(mc.model==m)&(mc.metal==metal)&(mc.noise_type=='baseline')]['MAPE_mid'].mean() for m in models]
        nm = [mc[(mc.model==m)&(mc.metal==metal)&(mc.noise_type=='gaussian')]['MAPE_mid'].mean() for m in models]
        ax.bar(x-w/2, cm, w, color='lightgray', edgecolor='black', label='clean')
        ax.bar(x+w/2, nm, w, color=ACCENT[metal], label='gaussian@30')
        for i in range(len(models)):
            if cm[i] > 0:
                ax.text(x[i], max(cm[i], nm[i])+1, f"{(nm[i]-cm[i])/cm[i]*100:+.0f}%", ha='center', fontsize=8)
        ax.set_xticks(x); ax.set_xticklabels(models); ax.set_ylabel('MAPE_mid (%)')
        ax.set_title(f"{metal}: per-model augmentation"); ax.legend(fontsize=8)
    plt.suptitle("Multi-classifier confirmation"); plt.tight_layout(); savefig("fig11_multiclassifier")
else:
    print("multi_classifier_results.csv not present -> skipping fig11.")

## 15. Workflow & architecture diagrams (Fig 1 / Fig 2)
Schematic figures generated as code.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle

# ---- Fig 1: high-level workflow ----
STAGES = [('Electrochemical\nmeasurement', ['Cyclic voltammetry','Cd & Pb','2-1000 ppm'], '#dbe9f6','#2E6CA4'),
          ('Multi-scan\nacquisition', ['10 scans/file','drop conditioning','-> 9 x 180 pts'], '#e7f1d8','#5a8f29'),
          ('Noise\naugmentation', ['Gaussian/t/pink/','hetero (+combined)','SNR 5-30 dB'], '#fde2d0','#d97b2b'),
          ('Feature\nengineering', ['7 per-scan','mean/std/CV','-> 17-dim'], '#f6e6c5','#c79a2e'),
          ('ML\nquantification', ['Random Forest','stratified 5-fold','MAPE_mid, R2'], '#e3dcf0','#6b4fa0'),
          ('Asymmetric\noutcome', ['Cd: MAPE down ~60%','Pb: no benefit','(20 seeds, SNR=30)'], '#d8efe8','#2c8c74')]
fig, ax = plt.subplots(figsize=(16, 4.6)); ax.set_xlim(0,100); ax.set_ylim(0,100); ax.axis('off'); ax.grid(False)
n = len(STAGES); w = (100-6-(n-1)*2.6)/n; cx = []
for i,(t,b,fc,ec) in enumerate(STAGES):
    x = 3 + i*(w+2.6) + w/2; cx.append(x)
    ax.add_patch(FancyBboxPatch((x-w/2, 30), w, 46, boxstyle='round,pad=0.6,rounding_size=2.4',
                                lw=2, facecolor=fc, edgecolor=ec))
    ax.add_patch(Circle((x, 74), 3.4, facecolor=ec, edgecolor='white', lw=1.5))
    ax.text(x, 74, str(i+1), ha='center', va='center', color='white', fontsize=11, fontweight='bold')
    ax.text(x, 67, t, ha='center', va='top', fontsize=10.5, fontweight='bold', color=ec)
    ax.text(x, 52, '\n'.join(b), ha='center', va='top', fontsize=8.5, linespacing=1.5)
for i in range(n-1):
    ax.add_patch(FancyArrowPatch((cx[i]+w/2, 53), (cx[i+1]-w/2, 53), arrowstyle='-|>',
                                 mutation_scale=20, lw=2.2, color=STAGES[i+1][3]))
ax.text(50, 92, "Study workflow: noise augmentation for ML quantification of Cd / Pb",
        ha='center', fontsize=13, fontweight='bold')
plt.tight_layout(); savefig("fig1_workflow")

In [ ]:
# ---- Fig 2: pipeline architecture ----
fig, ax = plt.subplots(figsize=(11, 9)); ax.set_xlim(0,100); ax.set_ylim(0,100); ax.axis('off'); ax.grid(False)
def abox(x, y, w, h, title, lines, fc, ec):
    ax.add_patch(FancyBboxPatch((x-w/2, y-h/2), w, h, boxstyle='round,pad=0.5,rounding_size=2',
                                lw=1.6, facecolor=fc, edgecolor=ec))
    ax.text(x, y+h/2-2.6, title, ha='center', va='top', fontsize=10, fontweight='bold', color=ec)
    ax.text(x, y+h/2-6.4, '\n'.join(lines), ha='center', va='top', fontsize=8, linespacing=1.4)
    return (x, y, w, h)
def aarrow(a, b, c='#6b4fa0'):
    ax.add_patch(FancyArrowPatch((a[0], a[1]-a[3]/2), (b[0], b[1]+b[3]/2), arrowstyle='-|>',
                                 mutation_scale=14, lw=1.7, color=c))
boxes = [abox(50, 92, 70, 11, "Raw CV .xlsx", ["CADMIUM/ + TIMBAL/, 15 ppm x 2 metals",
                                               "10 scans/file -> 9 usable, 180 pts"], '#dbe9f6','#2E6CA4'),
         abox(50, 76, 70, 9, "build_dataset -> data_cache.pkl", ["1500 files; Cd/Pb peak windows"], '#e7f1d8','#5a8f29'),
         abox(50, 61, 70, 10, "Synthetic noise injection (pre-features)", ["4 single + 5 combined; peak-relative SNR"], '#fde2d0','#d97b2b'),
         abox(50, 46, 70, 10, "Per-scan features (7) -> aggregate (17)", ["SavGol(11,3)+quad baseline; mean/std/cv"], '#f6e6c5','#c79a2e'),
         abox(50, 31, 70, 10, "RF + stratified 5-fold CV", ["scaler fit on train only; bins 0/10/30/100/500/1100"], '#e3dcf0','#6b4fa0'),
         abox(50, 17, 70, 8, "Metrics + analyses", ["R2, MAPE bands; stats / importance / tuning / leakage / sim"], '#d8efe8','#2c8c74')]
for a, b in zip(boxes, boxes[1:]): aarrow(a, b)
ax.text(50, 99, "CEJA pipeline architecture", ha='center', fontsize=13, fontweight='bold')
plt.tight_layout(); savefig("fig2_architecture")

## 16. Conclusion & honest notes
- **Central result:** synthetic noise injected before feature extraction acts as effective
  augmentation for **Cd²⁺** (large, significant MAPE_mid reduction) but **not for Pb²⁺** — an
  asymmetry that persists across single/combined noise, under hyperparameter tuning, and in the
  leakage-safe protocol.
- **Mechanism:** dispersion (std/CV) features gain importance after augmentation; the simulation
  shows peak shape is the driver. The benefit comes from noisy multi-scan *acquisition* (protocol B),
  not classic train-only augmentation (protocol C collapses on clean test data).
- **Honesty:** every number above is computed from your real data in this run — nothing is
  hard-coded. Magnitudes depend on `N_SEEDS_*` and the run toggles; report only what you actually ran.
- **Artifacts:** all CSVs in `results/` and figures (PNG+PDF) in `figs/` under the working dir.